# Superlinked Embeddings

[Superlinked](https://superlinked.com) is a self-hosted inference engine (SIE) for embedding, reranking, and extraction. The `sie-llamaindex` package provides drop-in LlamaIndex components: `SIEEmbedding` for vector stores, `SIEMultiModalEmbedding` for image retrieval with CLIP / SigLIP / ColPali, and `SIENodePostprocessor` for reranking.

This notebook shows the basic usage with `SIEEmbedding`.

## Setup

You need a running SIE instance and the `sie-llamaindex` package. See the [Superlinked quickstart](https://superlinked.com/docs) for deployment options (Docker, GPU, etc.).

In [ ]:
%pip install --quiet sie-llamaindex llama-index

Start a local SIE server:

```bash
docker run -p 8080:8080 ghcr.io/superlinked/sie-server:default
```

## Dense embeddings

`SIEEmbedding` implements LlamaIndex's `BaseEmbedding` interface. Set it as the default embed model or use it directly.

In [ ]:
from llama_index.core import Settings
from sie_llamaindex import SIEEmbedding

Settings.embed_model = SIEEmbedding(
    base_url="http://localhost:8080",
    model_name="BAAI/bge-m3",
)

embedding = Settings.embed_model.get_text_embedding("Your text here")
print(len(embedding))  # 1024 for BAAI/bge-m3

Any of SIE's 85+ supported models works. Common picks: `BAAI/bge-m3` (1024 dims, strong all-round), `NovaSearch/stella_en_400M_v5` (1024 dims, high quality), `nomic-ai/nomic-embed-text-v2-moe` (768 dims, multilingual MoE), `intfloat/e5-large-v2` (instruction-tuned, SIE handles query vs document encoding automatically). See the [Model Catalog](https://superlinked.com/models) for the full list.

### With VectorStoreIndex

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from sie_llamaindex import SIEEmbedding

Settings.embed_model = SIEEmbedding(model_name="BAAI/bge-m3")

documents = [
    Document(text="Machine learning is a branch of artificial intelligence."),
    Document(text="Neural networks are inspired by biological neurons."),
    Document(text="Deep learning uses multiple layers of neural networks."),
    Document(text="Python is popular for machine learning development."),
]

index = VectorStoreIndex.from_documents(documents)
response = index.as_query_engine().query("What is deep learning?")
print(response)

### Async

Both sync and async methods are available:

```python
embedding = await embed_model.aget_text_embedding(text)
query_embedding = await embed_model.aget_query_embedding(query)
```

## Reranking

The `sie-llamaindex` package also ships `SIENodePostprocessor`, a `BaseNodePostprocessor` that reranks retrieved nodes. It works with both cross-encoder models and ColBERT / late-interaction models.

In [ ]:
from sie_llamaindex import SIENodePostprocessor

reranker = SIENodePostprocessor(
    base_url="http://localhost:8080",
    model="jinaai/jina-reranker-v2-base-multilingual",
    top_n=5,
)

query_engine = index.as_query_engine(
    node_postprocessors=[reranker],
    similarity_top_k=20,  # retrieve 20, rerank to 5
)

response = query_engine.query("What is machine learning?")

## Multimodal embeddings

`SIEMultiModalEmbedding` extends LlamaIndex's `MultiModalEmbedding` base class, enabling image embedding with models like CLIP, SigLIP, and ColPali. It plugs into LlamaIndex's multimodal pipelines (e.g. `MultiModalVectorStoreIndex`).

```python
from sie_llamaindex import SIEMultiModalEmbedding

embed_model = SIEMultiModalEmbedding(
    base_url="http://localhost:8080",
    model_name="openai/clip-vit-large-patch14",
)

# Images
image_embedding = embed_model.get_image_embedding("photo.jpg")

# Text (inherited from BaseEmbedding)
text_embedding = embed_model.get_text_embedding("A photo of a cat")
```

## Resources

- [`sie-llamaindex` on PyPI](https://pypi.org/project/sie-llamaindex/)
- [Superlinked on GitHub](https://github.com/superlinked/sie)
- [Superlinked docs](https://superlinked.com/docs)
- [Model Catalog](https://superlinked.com/models)